# Architectural Comparison — Overview

OpenCode vs OpenHands vs PI. Runnable companion to the markdown report in
`../` (start at `../README.md`).

This notebook: (1) states the core thesis, (2) verifies all three condensed
harnesses import and run, (3) prints the headline comparison tables.

> **Methodology:** documentation-first, then verified against implementation.
OpenHands runs its real loop in an external, non-vendored `agent-server`; those
internals are marked `[EXTERNAL]`.

## The one fundamental difference

All three are *a while-loop that calls a model, runs requested tools, feeds
results back, and repeats.* They differ in **where that loop runs** and **who owns
state around it**:

- **OpenCode** — in-process, Effect-structured, SQLite transcript + per-turn git snapshots. *Powerful safe local agent.*

- **OpenHands** — control plane + **external sandboxed agent-server**; event-sourced; push webhooks. *Secure multi-tenant scale.*

- **PI** — tiny plain-async loop; no built-in permissions/MCP/sub-agents; everything is an extension. *Minimal hackable core.*

In [ ]:
import os, sys

def find_repo_root():
    markers = {'mini-agent', 'pi', 'openhands_all'}
    d = os.path.abspath(os.getcwd())
    while True:
        try:
            if markers.issubset(set(os.listdir(d))):
                return d
        except OSError:
            pass
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    for cand in ('/repo', os.path.expanduser('~')):
        if os.path.isdir(os.path.join(cand, 'mini-agent')):
            return cand
    raise RuntimeError('repo root not found; set REPO manually')

REPO = find_repo_root()
print('REPO =', REPO)

In [ ]:
# Verify all three condensed harnesses are importable from the repo.
import importlib, sys
paths = {
    'OpenCode (miniagent)': ('mini-agent', 'miniagent'),
    'OpenHands (harness)':  ('openhands_all/agent_harness', 'harness'),
    'PI (agent_harness)':   ('pi/agent_harness', 'agent_harness'),
}
for label, (subdir, pkg) in paths.items():
    p = os.path.join(REPO, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)
    mod = importlib.import_module(pkg)
    print(f'{label:24} -> {pkg} OK ({os.path.dirname(mod.__file__)})')

## Headline comparison

Full tables live in `../07-comparison-tables.md`. A compact view:

In [ ]:
rows = [
    ('loop location',   'in-process',            'EXTERNAL agent-server',   'in-process'),
    ('runtime',         'TypeScript / Effect',   'Python control plane',    'TypeScript / async'),
    ('state store',     'SQLite (parts)',        'append-only event log',   'JSONL tree'),
    ('permission',      'allow/ask/deny engine', 'policy+LLM analyzer (EXT)','none (extension hook)'),
    ('tool exec',       'AI SDK / fibers',       'EXTERNAL sandbox',        'parallel in-proc'),
    ('sandbox',         'no (perm+snapshot)',    'yes (Docker/remote)',     'no (extension)'),
    ('sub-agents',      'yes (task tool)',       'yes (planning+ACP)',      'no (extension)'),
    ('MCP',             'full client+OAuth',     'consumes+hosts proxy',    'none (by design)'),
    ('optimizes for',   'safe local power',      'secure cloud scale',      'minimal hackability'),
]
w = [16, 24, 26, 22]
hdr = ['dimension', 'OpenCode', 'OpenHands', 'PI']
print(' | '.join(h.ljust(w[i]) for i, h in enumerate(hdr)))
print('-+-'.join('-'*w[i] for i in range(4)))
for r in rows:
    print(' | '.join(str(c).ljust(w[i]) for i, c in enumerate(r)))

Next: open `01_opencode.ipynb`, `02_openhands.ipynb`, `03_pi.ipynb`.